# 04: LoRA Fine-tune (H3 Stretch Goal)

Trains a LoRA adapter for LLaVA-1.6-7B on the synthetic instruction-image
corpus. Configured for **A100 (40 GB)** — full `max_seq_len=1024` and the
default batch sizes. If Pro+ assigns you a T4 instead, reduce `max_seq_len`
to 768 and enable `gradient_checkpointing` (commented hint in the train cell).

**Estimated wallclock:** ~20-30 min on A100 (3 epochs, 200 examples).
**Estimated compute units:** ~6-7 on A100.

## Pro+ tip

Enable **'Run after disconnecting'** (Tools → Settings) and you can fire
this off, close the laptop, and come back to a finished adapter.

In [ ]:
# Mount Google Drive. The first run prompts for OAuth; subsequent runs
# in the same session reuse the token automatically.
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/grounded_vla')
ROOT.mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)

In [ ]:
# Verify accelerator. Pro+ should give you A100 most of the time; if you
# see T4, change Runtime -> Change runtime type -> A100 GPU and re-run.
import subprocess
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv'])

In [ ]:
# Clone (first run) or pull (re-run / re-attach) the repo into ephemeral /content.
# This ensures every session picks up the latest fixes from GitHub.
REPO_URL = 'https://github.com/<your-org>/grounded_vla.git'
import os, subprocess
if not os.path.exists('/content/grounded_vla'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/grounded_vla'], check=True)
else:
    subprocess.run(['git', '-C', '/content/grounded_vla', 'pull'], check=True)
%cd /content/grounded_vla

In [ ]:
# Install repo + GPU stack. Quiet to keep the cell output sane.
!pip install -q -e .[gpu,data]

In [ ]:
import os
WEIGHTS = '/content/drive/MyDrive/grounded_vla/hf_cache'
DATA    = '/content/drive/MyDrive/grounded_vla/data'
os.environ['HF_HOME'] = WEIGHTS
os.environ['TRANSFORMERS_CACHE'] = WEIGHTS
os.environ['TRANSFORMERS_OFFLINE'] = '1'

In [ ]:
# Train. The synthetic_sample fixture is tiny (3 examples) and only useful
# as a pipeline check; for real H3 numbers, swap to your real synthetic.jsonl
# (~200 reviewed examples). Edit JSONL/IMGS below.
from grounded_vla.lora import train_lora, LoRAConfig
from pathlib import Path

JSONL = f'{DATA}/synthetic_sample/synthetic_sample.jsonl'
IMGS  = f'{DATA}/synthetic_sample'
OUT   = Path('/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1')
OUT.mkdir(parents=True, exist_ok=True)

cfg = LoRAConfig(
    base_model=f'{WEIGHTS}/llava-v1.6-mistral-7b-hf',
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    learning_rate=2e-4,
    per_device_batch_size=1,
    gradient_accumulation_steps=8,
    num_epochs=3,
    max_seq_len=1024,  # A100 can handle full length; drop to 768 on T4
)
# T4 fallback: uncomment if Pro+ gave you a T4 instead of A100
# cfg.max_seq_len = 768
# from grounded_vla.lora import LoRAConfig as _C  # (gradient_checkpointing hook is in lora.py if needed)

train_lora(jsonl_path=JSONL, images_dir=IMGS, output_dir=OUT, config=cfg)

In [ ]:
# Verify adapter saved.
import os
for f in sorted(os.listdir('/content/drive/MyDrive/grounded_vla/checkpoints/llava-lora-r1')):
    print(f)

## Done

Adapter is on Drive. Move on to `05_eval_with_lora.ipynb` for the H3 ablation.